<a href="https://colab.research.google.com/github/LionardoGomiz/Apache-Spark/blob/L/PracticaT_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -y install openjdk-17-jdk-headless

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] += ":/usr/lib/jvm/java-17-openjdk-amd64/bin"

import subprocess
print(subprocess.check_output(["java","-version"], stderr=subprocess.STDOUT).decode())
print("JAVA_HOME =", os.environ["JAVA_HOME"])

In [209]:
!pip install -q findspark

In [210]:
!pip install -q pyspark

# REPORTE - ETL + ANÁLISIS DE E-COMERCE CON Pyspark
### Dataset: synthetic_ecommerce_sales_2025.csv

### Estructura:

1. Config general: SparkSession, logging, AQC
2. Helpers de observabilidad: log_df, observe metrics, etc
3. Lectura (Transformaciones) + Auditoría inicial (Acciones)
4. ETL
5. RFM / Top-N
6. Cohortes mensuales


#1.CONFIGURACION INICIAL

In [211]:
import findspark
findspark.init()

from pyspark.sql import SparkSession, functions as F, types as T, Window

spark = (SparkSession
         .builder
         .appName("Reporte-Ecommerce_Pyspark")
         .getOrCreate()
)

#Ajustamos el nivel de log para la consola
spark.sparkContext.setLogLevel("WARN")

#Activando AQE (Adapative Query Execution)
spark.conf.set("spark.sql.adaptive.enabled", "true")
print(f"AQE habilitado {spark.conf.get("spark.sql.adaptive.enabled")}")


AQE habilitado true


# 2. HELPERS DE OBSERVABILIDAD

In [212]:
from time import perf_counter

In [213]:
def log_df(df, nombre, sample=5, cols_nullcheck=("order_id","customer_id")):
  print(f"\n------{nombre}------")
  print(">> Plan (logical : physical):")
  df.explain(mode="extended")

  print(">> Esquema (tipos):")
  df.printSchema()

  t0 = perf_counter()
  total = df.count()
  print(f">> Filas: {total} (count en {perf_counter()-t0:.2f}s)")

  if cols_nullcheck:
    exprs = [F.sum(F.when(F.col(c).isNull(),1).otherwise(0)).alias(f"nulls_{c}")
  for c in cols_nullcheck]

  print(">> Muestra de filas: ")
  df.show(sample, truncate=False)




In [214]:
from pyspark.sql import Observation

In [215]:
def observe_metrics(df, name="obs"):
  obs = Observation(name)
  observed_df = df.observe(
      obs,
      F.count("*").alias("rows"),
      F.sum("revenue").alias("Sum_revenue"),
      F.avg("revenue").alias("Avg_revenue")
    )
  return observed_df, obs

In [216]:
spark.sparkContext.setCheckpointDir("out/_chk")

In [217]:
def mybe_checkpoint(df, do_it=False):
  return df.checkpoint(eager=True) if do_it else df

#3. LECTURA + AUDITORIA INICAL

In [218]:
#Definimos un esquema explicito
schema = T.StructType([
    T.StructField("order_id",          T.StringType(), True),
    T.StructField("order_date",        T.StringType(), True),
    T.StructField("customer_id",       T.StringType(), True),
    T.StructField("price",     T.StringType(), True),
    T.StructField("product_category",  T.StringType(), True),
    T.StructField("category",          T.StringType(), True),
    T.StructField("quantity",          T.StringType(), True),
    T.StructField("discount_rate",     T.StringType(), True),
    T.StructField("channel",           T.StringType(), True),
    T.StructField("country",           T.StringType(), True),
])


In [219]:
# Lectura del CSV

df_raw = (spark.read
        .option("header", True)
        .schema(schema)
        .csv("./data/synthetic_ecommerce_sales_2025.csv"))


In [220]:
df_raw = spark.read.csv('./data/synthetic_ecommerce_sales_2025.csv', header=True, inferSchema=True)

In [221]:
# Auditoría inicial (ACCIONES)
log_df(
    df_raw,
    "DF RAW (tal cual llega)",
    cols_nullcheck=(
        "order_id",
        "customer_id",
        "price",
        "quantity",
        "discount_rate"))



------DF RAW (tal cual llega)------
>> Plan (logical : physical):
== Parsed Logical Plan ==
Relation [order_id#2643,customer_id#2644,product_category#2645,product_price#2646,quantity#2647,order_date#2648,region#2649,payment_method#2650,delivery_days#2651,is_returned#2652,customer_rating#2653,discount_percent#2654,revenue#2655] csv

== Analyzed Logical Plan ==
order_id: int, customer_id: string, product_category: string, product_price: double, quantity: int, order_date: date, region: string, payment_method: string, delivery_days: int, is_returned: int, customer_rating: double, discount_percent: int, revenue: double
Relation [order_id#2643,customer_id#2644,product_category#2645,product_price#2646,quantity#2647,order_date#2648,region#2649,payment_method#2650,delivery_days#2651,is_returned#2652,customer_rating#2653,discount_percent#2654,revenue#2655] csv

== Optimized Logical Plan ==
Relation [order_id#2643,customer_id#2644,product_category#2645,product_price#2646,quantity#2647,order_date

# 4. ETL: Limpieza + Enriquecimiento

In [231]:
# Cast y normalizaciones
df_raw = df_raw.withColumn("revenue", F.col("product_price") * (1 - F.col("discount_percent")) * F.col("quantity"))


In [232]:
df_raw = df_raw.filter(
    (F.col("product_price") > 0) & (F.col("quantity") > 0))


In [233]:
df_raw = df_raw.withColumn("year", F.year("order_date")) \
               .withColumn("month", F.month("order_date"))


In [234]:
# Auditoria post-ETL (ACCIONES)
log_df(df_raw,
        "DF LIMPIO (cast + revenue + filtros)",
     cols_nullcheck=("order_date", "revenue", "channel"))


------DF LIMPIO (cast + revenue + filtros)------
>> Plan (logical : physical):
== Parsed Logical Plan ==
'Project [order_id#2643, customer_id#2644, product_category#2645, product_price#2646, quantity#2647, order_date#2648, region#2649, payment_method#2650, delivery_days#2651, is_returned#2652, customer_rating#2653, discount_percent#2654, revenue#2799, year#2813, month('order_date) AS month#2828]
+- Project [order_id#2643, customer_id#2644, product_category#2645, product_price#2646, quantity#2647, order_date#2648, region#2649, payment_method#2650, delivery_days#2651, is_returned#2652, customer_rating#2653, discount_percent#2654, revenue#2799, year(order_date#2648) AS year#2813]
   +- Filter ((product_price#2646 > cast(0 as double)) AND (quantity#2647 > 0))
      +- Project [order_id#2643, customer_id#2644, product_category#2645, product_price#2646, quantity#2647, order_date#2648, region#2649, payment_method#2650, delivery_days#2651, is_returned#2652, customer_rating#2653, discount_perc